In [ ]:
import torch

In [ ]:
def generate_drift_challenge():
    torch.manual_seed(2026)
    # Symulujemy 1000 słów w słowniku, embeddingi o wymiarze 32
    # Przestrzeń A (Technologiczna)
    space_A = torch.randn(1000, 32)

    # Przestrzeń B (Kulinarna) - zrotowana losowo i lekko zaszumiona
    random_rotation = torch.linalg.qr(torch.randn(32, 32))[0] # Losowa macierz ortogonalna
    space_B = torch.mm(space_A, random_rotation) + torch.randn(1000, 32) * 0.05

    # Słowa kotwiczne (indeksy 0 do 50) - ich znaczenie jest identyczne w obu bazach
    # Celowo nic z nimi nie robimy, posłużą do wyliczenia rotacji

    # Wstrzykujemy DRYT SEMANTYCZNY: słowa o indeksach 500, 600, 700 drastycznie zmieniają pozycję w Przestrzeni B
    space_B[500] = torch.randn(32)
    space_B[600] = torch.randn(32)
    space_B[700] = torch.randn(32)

    torch.save(space_A, "space_A.pt")
    torch.save(space_B, "space_B.pt")
    print("Wygenerowano przestrzenie wektorowe: space_A.pt oraz space_B.pt")

if __name__ == "__main__":
    generate_drift_challenge()

Wygenerowano przestrzenie wektorowe: space_A.pt oraz space_B.pt


In [ ]:
with open("space_A.pt", "rb") as f:
  space_a = torch.load(f)

with open("space_B.pt", "rb") as f:
  space_b = torch.load(f)

anchor_indices = space_a[:50]

embed_to_idx = {x:i for i, x in enumerate(anchor_indices)}

anchors = torch.tensor(list(embed_to_idx.values()))

#print(anchors)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
        36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49])


In [ ]:
def align_spaces(space_A, space_B, anchor_indices):
  A_anc = space_A[anchor_indices]
  B_anc = space_B[anchor_indices]

  M = A_anc.T @ B_anc
  U, S, Vt = torch.linalg.svd(M)
  R = torch.mm(U, Vt)

  return torch.mm(space_A, R)


space_A_aligned = align_spaces(space_a, space_b, anchors)

In [ ]:
A = space_A_aligned[:1000]
B = space_b[:1000]

A = A / A.norm(dim=1, keepdim=True)
B = B / B.norm(dim=1, keepdim=True)

cos_sim = (A*B).sum(dim=1)

najwieksze = torch.topk(cos_sim, 3, largest=False)

print(najwieksze)

torch.return_types.topk(
values=tensor([-0.2824,  0.1421,  0.3241]),
indices=tensor([500, 700, 600]))
